<h2>Encode high-cardinality categorical variables using advanced techniques such as target encoding, frequency encoding, or embedding layers for deep learning models, ensuring reduction in dimensionality without losing information.</h2>

In [2]:
import pandas as pd
import numpy as np

In [3]:
np.random.seed(42)
n_samples = 1000

In [4]:
cities = [f"City_{i}" for i in range(100)]

In [5]:
df = pd.DataFrame({
    'city': np.random.choice(cities, size=n_samples),
    'feature_val': np.random.normal(10, 2, size=n_samples)
})

In [6]:
df['target'] = np.random.binomial(n=1, p=0.3, size=n_samples)

In [7]:
print("Original Data Shape:", df.shape)
print("Unique Categories in 'city':", df['city'].nunique())
df.head()

Original Data Shape: (1000, 3)
Unique Categories in 'city': 100


,city,feature_val,target
0,City_51,13.046248,0
1,City_92,11.077820,0
2,City_14,7.925508,0
3,City_71,9.619323,0
4,City_60,8.248763,0


In [8]:
freq_map = df['city'].value_counts(normalize=True).to_dict()

In [9]:
df['city_freq_encoded'] = df['city'].map(freq_map)

In [10]:
print(df[['city', 'city_freq_encoded']].head())

      city  city_freq_encoded
0  City_51              0.012
1  City_92              0.011
2  City_14              0.010
3  City_71              0.008
4  City_60              0.012


In [12]:
global_mean = df['target'].mean()

In [13]:
target_stats = df.groupby('city')['target'].agg(['count', 'mean'])

In [14]:
smoothing = 10.0
smoothed_encodings = (target_stats['count'] * target_stats['mean'] + smoothing * global_mean) / (target_stats['count'] + smoothing)

In [15]:
df['city_target_encoded'] = df['city'].map(smoothed_encodings)

In [16]:
print(df[['city', 'target', 'city_target_encoded']].head())

      city  target  city_target_encoded
0  City_51       0             0.273182
1  City_92       0             0.286190
2  City_14       0             0.300500
3  City_71       0             0.278333
4  City_60       0             0.227727


In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import TruncatedSVD

In [20]:
encoder = OneHotEncoder(sparse_output=False)
city_ohe = encoder.fit_transform(df[['city']])

In [21]:
n_components = 5
svd = TruncatedSVD(n_components=n_components, random_state=42)
city_embeddings = svd.fit_transform(city_ohe)

In [22]:
for i in range(n_components):
    df[f'city_embed_{i+1}'] = city_embeddings[:, i]

In [23]:
print(f"Original Category High-Dim Representation: {city_ohe.shape[1]} One-Hot Columns")

Original Category High-Dim Representation: 100 One-Hot Columns


In [24]:
print(f"Compressed Dense Embedding Shape: {city_embeddings.shape}")

Compressed Dense Embedding Shape: (1000, 5)


In [25]:
print(df[['city'] + [f'city_embed_{i+1}' for i in range(n_components)]].head())

      city  city_embed_1  city_embed_2  city_embed_3  city_embed_4  \
0  City_51     -0.006841      0.004422     -0.021691     -0.028415   
1  City_92     -0.000298     -0.008710     -0.012741     -0.026222   
2  City_14     -0.001181     -0.010564      0.000525      0.004205   
3  City_71      0.000545      0.003441     -0.000264      0.000252   
4  City_60     -0.015363     -0.001581     -0.034484      0.010984   

   city_embed_5  
0     -0.001623  
1      0.016409  
2      0.000747  
3      0.000413  
4      0.021476  
